In [0]:
%sql
use catalog investment_pyspark;

In [0]:
df_bronze = spark.read.table("investment_pyspark.bronze.tradebook_raw")
df_bronze.show()

In [0]:
df_bronze.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType,IntegerType,BooleanType,LongType

In [0]:
df_silver = (
    df_bronze
    .withColumn("symbol",F.upper(F.trim(F.col("symbol"))))
    .withColumn("isin",F.upper(F.trim(F.col("isin"))))
    .withColumn("trade_date",F.to_timestamp(F.col("trade_date")))
    .withColumn("exchange",F.upper(F.trim(F.col("exchange"))))
    .withColumn("segment",F.upper(F.trim(F.col("segment"))))
    .withColumn("series",F.upper(F.trim(F.col("series"))))
    .withColumn("trade_type",F.upper(F.trim(F.col("trade_type"))))
    .withColumn("auction",F.col("auction").cast(BooleanType()))
    .withColumn("quantity",F.col("quantity").cast("double").cast(IntegerType()))
    .withColumn("price",F.col("price").cast(DecimalType(12,2)))
    .withColumn("trade_id",F.col("trade_id").cast(LongType()))
    .withColumn("order_id",F.col("order_id").cast(LongType()))
    .withColumn("order_execution_time",F.to_timestamp(F.col("order_execution_time"),"yyyy-MM-dd'T'HH:mm:ss"))
    .withColumn("bronze_time",F.to_timestamp(F.col("_inserted_timestamp")))
    .withColumn("Silver_timestamp",F.current_timestamp())
    .filter(
        F.col("symbol").isNotNull()
        & F.col("trade_date").isNotNull()
        & F.col("quantity").isNotNull()
        & F.col("price").isNotNull()
        & F.col("trade_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("order_execution_time").isNotNull()
        )
    .select("symbol","isin","trade_date","exchange","segment","series","trade_type","auction","quantity","price","trade_id","order_id","order_execution_time","Silver_timestamp")
)

In [0]:
df_silver.printSchema()
df_silver.show()

In [0]:
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("investment_pyspark.silver.tradebook_cleaned")

In [0]:
%sql
select * from investment_pyspark.silver.tradebook_cleaned;